# Fine-tuning eficiente com QLoRA — Roteador de sentimentos EscutIA

Uso rápido: selecione uma GPU no Colab e clique em **Runtime > Run all**. O notebook prepara o ambiente, valida os dados, treina o adapter QLoRA, testa o resultado e compara a execução com LoRA.

O treinamento fica ativado por padrão. Se quiser somente preparar e validar o ambiente, mude `EXECUTE_TRAINING` para `False` antes de executar tudo.

## O que será demonstrado

O fluxo mostra a diferença prática entre LoRA e QLoRA: o modelo-base é carregado em 4 bits com NF4 e somente um pequeno adapter é treinado. Assim, reduzimos a memória necessária sem alterar os pesos originais do modelo.

Durante a execução, o notebook também registra a configuração, a revisão exata do modelo, o gate de qualidade do dataset e os artefatos gerados. A comparação final usa o mesmo formato de saída e o mesmo conjunto congelado da etapa LoRA.

O modelo continua sendo um roteador: a saída esperada é um JSON com `sentimento`, não uma resposta conversacional final.

In [ ]:
# Verifique no Colab: Runtime > Change runtime type > GPU
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memória total (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    raise RuntimeError('GPU NVIDIA/CUDA não encontrada. Ative uma GPU no runtime do Colab antes de continuar.')

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/michaeldouglas/alura-llama-factory.git'
REPO_REF = 'feature/parte-3-fine-tuning-qlora'
REPO_DIR = Path('/content/alura-llama-factory')

# A branch precisa estar publicada no GitHub antes de executar esta célula.
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    print(f'Repositório já existe em {REPO_DIR}; clone não repetido.')

PROJECT_DIR = REPO_DIR / 'EscutIA'
QLORA_DIR = PROJECT_DIR / 'fine_tuning_qlora'
DATASET_DIR = PROJECT_DIR / 'dataset' / 'dados' / 'preparados'
GATE_PATH = PROJECT_DIR / 'dataset' / 'dados' / 'relatorios' / '11_validacao_final.json'
CONFIG_PATH = QLORA_DIR / 'configs' / 'qlora_escutia.yaml'
OUTPUT_DIR = QLORA_DIR / 'outputs' / 'resultados' / 'qlora_escutia_router'

required_paths = [DATASET_DIR / name for name in [
    'dataset_info.json', 'escutia_train.json', 'escutia_validation.json', 'escutia_evaluation.json'
]] + [GATE_PATH, CONFIG_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Arquivos obrigatórios ausentes após o clone:\n- ' + '\n- '.join(missing))

os.chdir(QLORA_DIR)
print('Projeto:', PROJECT_DIR)
print('Dataset:', DATASET_DIR)
print('Configuração:', CONFIG_PATH)
print('Saída:', OUTPUT_DIR)

In [ ]:
# Instala primeiro o runtime de treinamento e, por último, o LLaMA-Factory sem puxar a camada web/API.
!pip install -q -r /content/alura-llama-factory/EscutIA/fine_tuning_qlora/requirements-colab.txt
!pip install -q --no-deps llamafactory==0.9.5
print('Dependências instaladas. Se o Colab solicitar, reinicie o runtime e reexecute as células de verificação.')

In [ ]:
import json
from collections import Counter

gate = json.loads(GATE_PATH.read_text(encoding='utf-8'))
decision = gate.get('decisao')
print('Gate do dataset:', decision)
if decision != 'DATA_READY_FOR_SFT':
    raise RuntimeError('Treinamento bloqueado: o dataset não está marcado como DATA_READY_FOR_SFT.')

def load_json_rows(path):
    value = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(value, list) or not value:
        raise ValueError(f'Dataset vazio ou inválido: {path}')
    return value

def label(row):
    output = row.get('output', {})
    if isinstance(output, str):
        output = json.loads(output)
    value = output.get('sentimento')
    if value not in {'positivo', 'neutro', 'negativo'}:
        raise ValueError(f'Rótulo inválido: {value!r}')
    return value

train_rows = load_json_rows(DATASET_DIR / 'escutia_train.json')
validation_rows = load_json_rows(DATASET_DIR / 'escutia_validation.json')
evaluation_rows = load_json_rows(DATASET_DIR / 'escutia_evaluation.json')
print('Treino:', len(train_rows), Counter(label(row) for row in train_rows))
print('Validação:', len(validation_rows), Counter(label(row) for row in validation_rows))
print('Avaliação congelada:', len(evaluation_rows), Counter(label(row) for row in evaluation_rows))
print('Arquivos preparados:', sorted(path.name for path in DATASET_DIR.iterdir()))

In [ ]:
# Inspeção final do YAML antes de qualquer execução.
import yaml

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
required_config = {
    'model_name_or_path', 'model_revision', 'quantization_method', 'quantization_bit',
    'quantization_type', 'double_quantization', 'stage', 'finetuning_type',
    'dataset_dir', 'dataset', 'eval_dataset', 'template', 'output_dir'
}
missing_config = sorted(required_config - set(config))
if missing_config:
    raise ValueError(f'Campos ausentes na configuração: {missing_config}')
if config['quantization_bit'] != 4 or config['quantization_method'] != 'bnb':
    raise ValueError('A configuração desta etapa deve usar bitsandbytes em 4 bits.')
if config['finetuning_type'] != 'lora' or config['stage'] != 'sft':
    raise ValueError('A etapa esperada é SFT com adapter LoRA.')
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(f'Saída já contém artefatos; não sobrescrever: {OUTPUT_DIR}')

print(CONFIG_PATH.read_text(encoding='utf-8'))
print('Configuração validada sem iniciar treinamento.')

## Execução

As células anteriores conferem a GPU, clonam a versão correta do projeto, instalam o runtime de treinamento, verificam o dataset e validam o YAML.

A célula seguinte inicia o treinamento quando `EXECUTE_TRAINING` está como `True`. Ela mede a duração e acompanha o uso de memória da GPU para alimentar a comparação final. Use `False` se quiser somente preparar e validar o ambiente.

In [ ]:
EXECUTE_TRAINING = True

if not EXECUTE_TRAINING:
    training_runtime_seconds = None
    gpu_memory_peak_mib_sampled = None
    training_started_utc = None
    training_finished_utc = None
    print('Treinamento desativado. O ambiente e o dataset foram somente preparados e validados.')
else:
    import threading
    import time
    from datetime import datetime, timezone

    def sample_gpu_memory(stop_event, samples):
        while not stop_event.is_set():
            try:
                result = subprocess.run(
                    ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                    capture_output=True, text=True, check=True
                )
                values = [int(line.strip()) for line in result.stdout.splitlines() if line.strip().isdigit()]
                if values:
                    samples.append(max(values))
            except (FileNotFoundError, subprocess.CalledProcessError, ValueError):
                pass
            stop_event.wait(1.0)

    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    command = ['llamafactory-cli', 'train', str(CONFIG_PATH)]
    gpu_memory_samples = []
    sampler_stop = threading.Event()
    sampler = threading.Thread(target=sample_gpu_memory, args=(sampler_stop, gpu_memory_samples), daemon=True)
    training_started_utc = datetime.now(timezone.utc).isoformat()
    started = time.perf_counter()
    sampler.start()
    print('Executando:', ' '.join(command), flush=True)
    try:
        training_environment = os.environ.copy()
        training_environment['PYTHONUNBUFFERED'] = '1'
        subprocess.run(command, cwd=QLORA_DIR, env=training_environment, check=True)
    finally:
        training_runtime_seconds = time.perf_counter() - started
        training_finished_utc = datetime.now(timezone.utc).isoformat()
        sampler_stop.set()
        sampler.join(timeout=2)
        gpu_memory_peak_mib_sampled = max(gpu_memory_samples) if gpu_memory_samples else None
    print('Treinamento concluído. Confira os artefatos em:', OUTPUT_DIR)
    print(f'Duração observada: {training_runtime_seconds / 60:.2f} minutos')
    print(f"Pico de memória GPU amostrado: {gpu_memory_peak_mib_sampled or 'não registrado'} MiB")

In [ ]:
# Registro local do experimento; execute após o treinamento.
from datetime import datetime, timezone
import hashlib

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not OUTPUT_DIR.exists():
    print('Ainda não há output para registrar.')
else:
    adapter_files = sorted(path for path in OUTPUT_DIR.rglob('*') if path.is_file())
    manifest = {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_name_or_path': config['model_name_or_path'],
        'model_revision': config['model_revision'],
        'finetuning_type': config['finetuning_type'],
        'quantization_method': config['quantization_method'],
        'quantization_bit': config['quantization_bit'],
        'dataset_gate': decision,
        'dataset_train_records': len(train_rows),
        'dataset_validation_records': len(validation_rows),
        'training_started_utc': training_started_utc,
        'training_finished_utc': training_finished_utc,
        'training_runtime_seconds_observed': training_runtime_seconds,
        'gpu_memory_peak_mib_sampled': gpu_memory_peak_mib_sampled,
        'output_dir': str(OUTPUT_DIR),
        'files': [{'path': str(path.relative_to(OUTPUT_DIR)), 'sha256': sha256(path)} for path in adapter_files]
    }
    manifest_path = QLORA_DIR / 'outputs' / 'qlora_run_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Manifesto salvo em:', manifest_path)
    print('Arquivos gerados:', len(adapter_files))

## Inferência pós-treinamento

Depois que o treinamento termina, esta etapa verifica se o adapter foi criado e o aplica ao mesmo modelo-base. O objetivo agora não é treinar novamente, mas observar como o roteador responde a textos novos.

Como tudo acontece no mesmo Colab, o modelo já está no cache e o adapter está no diretório de saída. Por isso, esta etapa não faz clone, instalação ou novo download.

In [ ]:
import re
from contextlib import nullcontext

ADAPTER_DIR = OUTPUT_DIR
EVALUATION_DIR = QLORA_DIR / 'outputs' / 'avaliacao'
if not (ADAPTER_DIR / 'adapter_config.json').exists():
    raise FileNotFoundError(f'Adapter não encontrado em {ADAPTER_DIR}. O treinamento precisa terminar antes desta etapa.')
if not torch.cuda.is_available():
    raise RuntimeError('GPU NVIDIA/CUDA não encontrada para a inferência QLoRA.')

MODEL_NAME = config['model_name_or_path']
MODEL_REVISION = config['model_revision']
QUANTIZATION_BITS = config['quantization_bit']
QUANTIZATION_TYPE = config['quantization_type']
DOUBLE_QUANTIZATION = config['double_quantization']
print('Adapter:', ADAPTER_DIR)
print('Modelo:', MODEL_NAME)
print('GPU:', torch.cuda.get_device_name(0))
print('Quantização:', f'{QUANTIZATION_BITS} bits / {QUANTIZATION_TYPE.upper()} / double quantization={DOUBLE_QUANTIZATION}')

### Carregar o modelo quantizado e o adapter

O QLoRA não cria um modelo completo: ele salva apenas as alterações aprendidas pelo adapter. Nesta etapa carregamos novamente o modelo-base em 4 bits, configuramos NF4 e aplicamos o adapter treinado sobre ele. O resultado é o modelo especializado que será usado nas inferências.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=QUANTIZATION_TYPE,
    bnb_4bit_use_double_quant=DOUBLE_QUANTIZATION,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
modelo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, quantization_config=quantization_config,
    device_map='auto', local_files_only=True
)
modelo = PeftModel.from_pretrained(modelo_base, ADAPTER_DIR, local_files_only=True)
modelo.eval()
MODEL_INPUT_DEVICE = next(modelo.parameters()).device
print('Modelo-base quantizado + adapter QLoRA carregados.')
print('Dispositivo de entrada:', MODEL_INPUT_DEVICE)

### Classificar textos

A função abaixo monta o prompt no formato conversacional usado pelo Qwen e envia o texto ao modelo. Depois da geração, ela extrai e valida o JSON, aceitando somente os rótulos `negativo`, `neutro` e `positivo`.

O parâmetro `usar_adapter` permite executar a mesma entrada com ou sem o adapter, o que ajuda a observar o efeito do fine-tuning.

In [ ]:
INSTRUCTION = 'Classifique o sentimento predominante do texto como negativo, neutro ou positivo e responda somente com um JSON válido no formato {\"sentimento\":\"<rotulo>\"}.'
SYSTEM_PROMPT = 'Você é um roteador de sentimentos. Responda somente com JSON válido no formato {\"sentimento\":\"negativo|neutro|positivo\"}.'
LABELS = ('negativo', 'neutro', 'positivo')

def extrair_json(resposta):
    trecho = re.search(r'\{.*?\}', resposta, flags=re.DOTALL)
    if trecho is None:
        return None
    try:
        candidato = json.loads(trecho.group(0))
    except json.JSONDecodeError:
        return None
    if set(candidato) != {'sentimento'} or candidato['sentimento'] not in LABELS:
        return None
    return candidato

def classificar_sentimento(texto, max_new_tokens=24, usar_adapter=True):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError('Informe um texto não vazio.')
    mensagens = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'{INSTRUCTION}\n\nTexto: {texto.strip()}'},
    ]
    prompt = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=int(config.get('cutoff_len', 256)))
    entradas = {nome: valor.to(MODEL_INPUT_DEVICE) for nome, valor in entradas.items()}
    contexto_adapter = nullcontext() if usar_adapter else modelo.disable_adapter()
    with contexto_adapter, torch.inference_mode():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    novos_tokens = saida[0, entradas['input_ids'].shape[1]:]
    resposta_bruta = tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()
    resposta_json = extrair_json(resposta_bruta)
    return {
        'texto': texto, 'resposta_bruta': resposta_bruta, 'resposta_json': resposta_json,
        'sentimento': resposta_json['sentimento'] if resposta_json else None,
        'modelo_avaliado': 'adapter_qlora' if usar_adapter else 'modelo_base',
    }

### Testar exemplos e texto próprio

Primeiro usamos alguns exemplos para confirmar rapidamente o comportamento do roteador. Em seguida, a variável `texto_teste` pode ser alterada para experimentar uma frase própria.

A resposta bruta mostra exatamente o que o modelo gerou; o campo JSON só é considerado válido depois da validação do formato.

In [ ]:
exemplos = [
    'Estou muito feliz com a solução que encontrei para o problema.',
    'A reunião aconteceu conforme o planejado.',
    'Estou preocupado e frustrado com o resultado da prova.',
    'Hoje recebi uma notícia inesperada e preciso pensar antes de responder.',
]
display(__import__('pandas').DataFrame([classificar_sentimento(texto) for texto in exemplos]))

texto_teste = 'Estou com raiva e não quero conversar com ninguém hoje.'
resultado_teste = classificar_sentimento(texto_teste)
print('Texto:', resultado_teste['texto'])
print('Resposta bruta:', resultado_teste['resposta_bruta'])
print('JSON para o LLM principal:', json.dumps(resultado_teste['resposta_json'], ensure_ascii=False) if resultado_teste['resposta_json'] else 'inválido')
print('Sentimento identificado:', resultado_teste['sentimento'] or 'não identificado')

## Avaliação e comparação com LoRA

Agora usamos o mesmo conjunto congelado da etapa LoRA, sem alterar seus exemplos, para que as métricas sejam comparáveis. Para cada texto, o notebook gera uma classificação QLoRA, verifica o JSON e compara o rótulo previsto com o esperado.

Além da acurácia, calculamos o F1 macro, que resume o desempenho nas três classes mesmo quando elas apresentam comportamentos diferentes. Também reunimos loss de validação, duração, tamanho do adapter e memória registrada. A execução LoRA histórica não registrou seu pico de memória; esse campo aparecerá como não registrado.

In [ ]:
import time
import pandas as pd
from sklearn.metrics import f1_score

ESCUTIA_DIR = QLORA_DIR.parent
DATASET_EVALUATION_FILE = ESCUTIA_DIR / 'dataset' / 'dados' / 'preparados' / 'escutia_evaluation.json'
LORA_DIR = ESCUTIA_DIR / 'fine_tuning_lora'
LORA_ADAPTER_DIR = LORA_DIR / 'outputs' / 'resultados' / 'lora_escutia_router'
LORA_EVALUATION_CSV = LORA_DIR / 'outputs' / 'avaliacao' / 'avaliacao_lora.csv'
QLORA_TRAIN_RESULTS = ADAPTER_DIR / 'train_results.json'
QLORA_EVAL_RESULTS = ADAPTER_DIR / 'eval_results.json'
QLORA_MANIFEST = QLORA_DIR / 'outputs' / 'qlora_run_manifest.json'
LORA_TRAIN_RESULTS = LORA_ADAPTER_DIR / 'train_results.json'
LORA_EVAL_RESULTS = LORA_ADAPTER_DIR / 'eval_results.json'
required_files = [DATASET_EVALUATION_FILE, LORA_EVALUATION_CSV, LORA_TRAIN_RESULTS, LORA_EVAL_RESULTS]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError('Arquivos necessários para a comparação ausentes:\n- ' + '\n- '.join(missing_files))

def metricas_classificacao(dataframe):
    predicoes = dataframe['predito'].fillna('__invalido__')
    return {
        'acuracia': float((dataframe['esperado'] == predicoes).mean()),
        'f1_macro': float(f1_score(dataframe['esperado'], predicoes, labels=list(LABELS), average='macro', zero_division=0)),
        'taxa_json_valido': float(dataframe['json_valido'].astype(bool).mean()),
        'amostras': int(len(dataframe)),
    }

def tamanho_adapter_bytes(adapter_dir):
    pesos = adapter_dir / 'adapter_model.safetensors'
    return pesos.stat().st_size if pesos.exists() else None

def ler_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

registros_congelados = ler_json(DATASET_EVALUATION_FILE)
inicio_avaliacao = time.perf_counter()
linhas_qlora = []
for indice, registro in enumerate(registros_congelados, start=1):
    predicao = classificar_sentimento(registro['input'], usar_adapter=True)
    esperado = json.loads(registro['output'])['sentimento'] if isinstance(registro['output'], str) else registro['output']['sentimento']
    linhas_qlora.append({'esperado': esperado, 'predito': predicao['sentimento'], 'json_valido': predicao['resposta_json'] is not None, 'resposta_bruta': predicao['resposta_bruta']})
    if indice == 1 or indice % 50 == 0 or indice == len(registros_congelados):
        print(f'Avaliados com QLoRA: {indice}/{len(registros_congelados)}')
tempo_avaliacao_qlora = time.perf_counter() - inicio_avaliacao
df_qlora_congelado = pd.DataFrame(linhas_qlora)
metricas_qlora = metricas_classificacao(df_qlora_congelado)

df_lora_congelado = pd.read_csv(LORA_EVALUATION_CSV)
if len(df_lora_congelado) != len(registros_congelados):
    raise ValueError('LoRA e QLoRA não foram avaliados sobre a mesma quantidade de amostras.')
metricas_lora = metricas_classificacao(df_lora_congelado)

qlora_train_results = ler_json(QLORA_TRAIN_RESULTS) if QLORA_TRAIN_RESULTS.exists() else {}
qlora_eval_results = ler_json(QLORA_EVAL_RESULTS) if QLORA_EVAL_RESULTS.exists() else {}
lora_train_results = ler_json(LORA_TRAIN_RESULTS)
lora_eval_results = ler_json(LORA_EVAL_RESULTS)
qlora_manifest = ler_json(QLORA_MANIFEST) if QLORA_MANIFEST.exists() else {}

comparacao = {
    'LoRA': {'modelo': 'Qwen/Qwen2.5-0.5B-Instruct', 'dispositivo': 'XPU (execução histórica)', 'memoria_gpu_pico_mib': None, 'duracao_treinamento_segundos': lora_train_results.get('train_runtime'), 'duracao_avaliacao_segundos': lora_eval_results.get('eval_runtime'), 'tamanho_adapter_bytes': tamanho_adapter_bytes(LORA_ADAPTER_DIR), 'loss_validacao': lora_eval_results.get('eval_loss'), **metricas_lora},
    'QLoRA': {'modelo': MODEL_NAME, 'dispositivo': torch.cuda.get_device_name(0), 'memoria_gpu_pico_mib': qlora_manifest.get('gpu_memory_peak_mib_sampled'), 'duracao_treinamento_segundos': qlora_train_results.get('train_runtime', qlora_manifest.get('training_runtime_seconds_observed')), 'duracao_avaliacao_segundos': tempo_avaliacao_qlora, 'tamanho_adapter_bytes': tamanho_adapter_bytes(ADAPTER_DIR), 'loss_validacao': qlora_eval_results.get('eval_loss'), **metricas_qlora},
}

tabela_comparacao = pd.DataFrame([
    {'métrica': 'Modelo-base', 'LoRA': comparacao['LoRA']['modelo'], 'QLoRA': comparacao['QLoRA']['modelo']},
    {'métrica': 'Dispositivo', 'LoRA': comparacao['LoRA']['dispositivo'], 'QLoRA': comparacao['QLoRA']['dispositivo']},
    {'métrica': 'Memória GPU máxima (MiB)', 'LoRA': comparacao['LoRA']['memoria_gpu_pico_mib'] or 'não registrada', 'QLoRA': comparacao['QLoRA']['memoria_gpu_pico_mib'] or 'não registrada'},
    {'métrica': 'Duração do treinamento (min)', 'LoRA': round((comparacao['LoRA']['duracao_treinamento_segundos'] or 0) / 60, 2), 'QLoRA': round((comparacao['QLoRA']['duracao_treinamento_segundos'] or 0) / 60, 2)},
    {'métrica': 'Tamanho do adapter (MiB)', 'LoRA': round((comparacao['LoRA']['tamanho_adapter_bytes'] or 0) / 2**20, 2), 'QLoRA': round((comparacao['QLoRA']['tamanho_adapter_bytes'] or 0) / 2**20, 2)},
    {'métrica': 'Loss de validação', 'LoRA': comparacao['LoRA']['loss_validacao'], 'QLoRA': comparacao['QLoRA']['loss_validacao']},
    {'métrica': 'Acurácia', 'LoRA': comparacao['LoRA']['acuracia'], 'QLoRA': comparacao['QLoRA']['acuracia']},
    {'métrica': 'F1 macro', 'LoRA': comparacao['LoRA']['f1_macro'], 'QLoRA': comparacao['QLoRA']['f1_macro']},
    {'métrica': 'Taxa de JSON válido', 'LoRA': comparacao['LoRA']['taxa_json_valido'], 'QLoRA': comparacao['QLoRA']['taxa_json_valido']},
])
display(tabela_comparacao)
COMPARISON_DIR = QLORA_DIR / 'outputs' / 'comparacao'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
(COMPARISON_DIR / 'comparacao_lora_qlora.json').write_text(json.dumps(comparacao, ensure_ascii=False, indent=2), encoding='utf-8')
tabela_comparacao.to_csv(COMPARISON_DIR / 'comparacao_lora_qlora.csv', index=False, encoding='utf-8')
print('Comparação salva em:', COMPARISON_DIR)

## Encerramento

Ao concluir, confira a tabela e os arquivos produzidos em `outputs/avaliacao` e `outputs/comparacao`. Eles mostram se o adapter melhorou a classificação, respeitou o contrato JSON e qual foi o custo computacional observado.

Salve os diretórios `outputs/resultados/qlora_escutia_router`, `outputs/avaliacao` e `outputs/comparacao` ou copie-os para o Google Drive antes de encerrar a sessão do Colab. O adapter QLoRA depende do modelo-base e não deve ser tratado como um modelo completo.

## Empacotar para o Hugging Face

A última célula monta um pacote limpo para publicação. Ela inclui o adapter, o tokenizer, a configuração, o manifesto e as métricas, mas deixa checkpoints e arquivos de otimização fora. Ao final, o Colab inicia o download de um único ZIP.

In [ ]:
import shutil
import zipfile
from google.colab import files

PACKAGE_DIR = QLORA_DIR / 'outputs' / 'huggingface' / 'escutia-qlora'
ZIP_PATH = QLORA_DIR / 'outputs' / 'escutia-qlora-huggingface.zip'

# Recria somente a pasta de empacotamento gerada pelo notebook; os resultados originais permanecem intactos.
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

required_files = [ADAPTER_DIR / 'adapter_config.json', ADAPTER_DIR / 'adapter_model.safetensors']
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError('Arquivos obrigatórios do adapter ausentes:\n- ' + '\n- '.join(missing_files))

files_to_package = {
    'adapter_config.json': ADAPTER_DIR / 'adapter_config.json',
    'adapter_model.safetensors': ADAPTER_DIR / 'adapter_model.safetensors',
    'tokenizer_config.json': ADAPTER_DIR / 'tokenizer_config.json',
    'tokenizer.json': ADAPTER_DIR / 'tokenizer.json',
    'special_tokens_map.json': ADAPTER_DIR / 'special_tokens_map.json',
    'added_tokens.json': ADAPTER_DIR / 'added_tokens.json',
    'chat_template.jinja': ADAPTER_DIR / 'chat_template.jinja',
    'merges.txt': ADAPTER_DIR / 'merges.txt',
    'vocab.json': ADAPTER_DIR / 'vocab.json',
    'qlora_escutia.yaml': CONFIG_PATH,
    'qlora_run_manifest.json': QLORA_DIR / 'outputs' / 'qlora_run_manifest.json',
    'train_results.json': ADAPTER_DIR / 'train_results.json',
    'eval_results.json': ADAPTER_DIR / 'eval_results.json',
    'avaliacao_qlora.json': QLORA_DIR / 'outputs' / 'avaliacao' / 'avaliacao_qlora.json',
    'avaliacao_qlora.csv': QLORA_DIR / 'outputs' / 'avaliacao' / 'avaliacao_qlora.csv',
    'comparacao_lora_qlora.json': QLORA_DIR / 'outputs' / 'comparacao' / 'comparacao_lora_qlora.json',
    'comparacao_lora_qlora.csv': QLORA_DIR / 'outputs' / 'comparacao' / 'comparacao_lora_qlora.csv',
}

for nome, origem in files_to_package.items():
    if origem.exists():
        shutil.copy2(origem, PACKAGE_DIR / nome)

model_card = f'''# EscutIA QLoRA

Adapter QLoRA para classificação de sentimentos em português.

## Modelo-base

- modelo: `{MODEL_NAME}`
- revisão: `{MODEL_REVISION}`
- quantização do treinamento: 4 bits, NF4, double quantization
- tarefa: classificação em `negativo`, `neutro` ou `positivo`

Este repositório contém um adapter, não um modelo completo. Para usar os pesos, carregue o modelo-base e aplique o adapter com PEFT.

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained('{MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained('{MODEL_NAME}')
model = PeftModel.from_pretrained(base, 'SEU_USUARIO/escutia-qlora')
```

Consulte os arquivos de avaliação e comparação incluídos neste pacote para conhecer o desempenho observado.
'''.strip() + '\n'
(PACKAGE_DIR / 'README.md').write_text(model_card, encoding='utf-8')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PACKAGE_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=f'escutia-qlora/{path.name}')

packaged_names = sorted(path.name for path in PACKAGE_DIR.iterdir() if path.is_file())
print('Pacote Hugging Face criado:', ZIP_PATH)
print('Arquivos incluídos:', ', '.join(packaged_names))
print('Checkpoints e arquivos de otimização não foram incluídos.')
files.download(str(ZIP_PATH))